# Lab 2.4: Exercise — Build Your Own Style Dataset

---

## 🎯 Learning Objective

In this exercise, you will:
1. **Create a small preference dataset** (5-10 examples) with a distinctive style
2. **Train two models**: Standard SFT and DITTO
3. **Compare their outputs** to understand when DITTO outperforms SFT

**Choose your persona:**
- 🏴‍☠️ **Pirate** — Arr matey! Respond like a salty sea dog!
- 🍕 **Perpetually Hungry Person** — Everything relates to food somehow
- 🤓 **Overly Enthusiastic Nerd** — EVERYTHING is fascinating!
- 📜 **Shakespearean** — Speak in iambic pentameter, forsooth!

---

## Setup

In [ ]:
# %%capture
# !pip install torch transformers accelerate peft trl datasets

In [ ]:
import os
import sys
import logging
import random
import warnings
import inspect
from typing import List, Dict, Any, Optional, Literal
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer,
    TrainingArguments,
)
from transformers.trainer_callback import TrainerCallback

from peft import PeftModel, get_peft_model, LoraConfig, TaskType
from peft.utils import ModulesToSaveWrapper
from peft.tuners.tuners_utils import BaseTunerLayer

from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Helper functions from Lab 2.3
def is_openai_format(messages):
    """Check if messages are in OpenAI format (list of dicts with 'role' and 'content')."""
    if not isinstance(messages, list):
        return False
    if len(messages) == 0:
        return False
    return all(
        isinstance(m, dict) and "role" in m and "content" in m 
        for m in messages
    )


def apply_chat_template(
    example,
    tokenizer,
    task: Literal["sft", "generation", "ditto", "dpo"]
):
    """
    Apply chat template to examples based on task type.
    From run_ditto.py - handles both SFT and DITTO/DPO formats.
    """
    if task in ["sft", "generation"]:
        messages = example["chosen"]
        example["text"] = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True if task == "generation" else False,
        )
        
    elif task in ["ditto", "dpo"]:
        if not is_openai_format(example["chosen"]):
            raise ValueError(
                f"Could not format example as dialogue for `{task}` task! Require OpenAI format"
            )
        
        if "prompt" in example and is_openai_format(example["prompt"]):
            prompt_messages = example["prompt"]
            chosen_messages = example["chosen"]
        else:
            prompt_messages = example["chosen"][:-1]
            chosen_messages = example["chosen"][-1:]

        example["text_prompt"] = tokenizer.apply_chat_template(prompt_messages, tokenize=False)
        example["text_chosen"] = tokenizer.apply_chat_template(chosen_messages, tokenize=False)
        
        if tokenizer.bos_token and example["text_chosen"] and example["text_chosen"].startswith(tokenizer.bos_token):
            example["text_chosen"] = example["text_chosen"][len(tokenizer.bos_token):]

        if task == "dpo" and "rejected" in example and example["rejected"]:
            if is_openai_format(example["rejected"]):
                if "prompt" in example and is_openai_format(example["prompt"]):
                    rejected_messages = example["rejected"]
                else:
                    rejected_messages = example["rejected"][-1:]
                
                example["text_rejected"] = tokenizer.apply_chat_template(rejected_messages, tokenize=False)
                
                if tokenizer.bos_token and example["text_rejected"] and example["text_rejected"].startswith(tokenizer.bos_token):
                    example["text_rejected"] = example["text_rejected"][len(tokenizer.bos_token):]
            else:
                example["text_rejected"] = example["rejected"]

    return example


def copy_adapter_weights(src_adapter_name, tgt_adapter_name, model):
    """
    Copy LoRA adapter weights from source adapter to target adapter.
    From run_ditto.py - used to initialize DITTO adapter from SFT.
    """
    lora_modules = [
        module for module in model.modules() 
        if isinstance(module, (BaseTunerLayer, ModulesToSaveWrapper))
    ]

    with torch.no_grad():
        for model_module in lora_modules:
            if hasattr(model_module, 'lora_A') and src_adapter_name in model_module.lora_A.keys():
                model_module.lora_A[tgt_adapter_name].load_state_dict(
                    model_module.lora_A[src_adapter_name].state_dict()
                )
                model_module.lora_B[tgt_adapter_name].load_state_dict(
                    model_module.lora_B[src_adapter_name].state_dict()
                )
    
            if hasattr(model_module, 'lora_embedding_A') and src_adapter_name in model_module.lora_embedding_A.keys():
                model_module.lora_embedding_A[tgt_adapter_name].load_state_dict(
                    model_module.lora_embedding_A[src_adapter_name].state_dict()
                )
                model_module.lora_embedding_B[tgt_adapter_name].load_state_dict(
                    model_module.lora_embedding_B[src_adapter_name].state_dict()
                )
    
    logger.info(f"Copied adapter weights: {src_adapter_name} → {tgt_adapter_name}")


def save_adapter(model, tokenizer, output_dir, adapter_name):
    """Save a specific adapter to disk."""
    os.makedirs(output_dir, exist_ok=True)
    model.set_adapter(adapter_name)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    if hasattr(model, 'config'):
        model.config.save_pretrained(output_dir)
    logger.info(f"Adapter '{adapter_name}' saved to {output_dir}")

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, threshold=1.0):
        self.threshold = threshold
    def on_step_begin(self, args, state, control, **kwargs):  
        if len(state.log_history) > 0:
            for k in state.log_history[::-1]:
                if "loss" in k and k["loss"] < self.threshold:
                    control.should_training_stop = True
                    break

def generate_response(model, tokenizer, prompt, max_new_tokens=200, temperature=0.7):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# Detect TRL version for API compatibility (same as run_ditto.py)
def _get_trainer_tokenizer_kwarg():
    """
    Newer TRL versions (>=0.12) use 'processing_class' instead of 'tokenizer'.
    Returns the correct keyword argument name.
    """
    sig = inspect.signature(DPOTrainer.__init__)
    if 'processing_class' in sig.parameters:
        return 'processing_class'
    elif 'tokenizer' in sig.parameters:
        return 'tokenizer'
    else:
        return 'processing_class'

TOKENIZER_KWARG = _get_trainer_tokenizer_kwarg()
print(f"TRL trainer tokenizer kwarg: {TOKENIZER_KWARG}")

---

## 📝 TASK 1: Create Your Style Dataset

Create **5-10 high-quality demonstrations** that exemplify your chosen style.

**Tips for good demonstrations:**
- Be **consistent** — same style markers across all examples
- Cover **diverse topics** — helps generalization
- Make style **distinctive** — clear patterns the model can learn
- Include **signature elements** — catchphrases, emoji, etc.

In [ ]:
# ================================================================
# EXAMPLE: PIRATE PERSONA 🏴‍☠️
# ================================================================

PIRATE_DEMONSTRATIONS = [
    {
        "prompt": "How do I make coffee?",
        "response": """Ahoy! Ye want to brew the black gold, do ye? Here be the captain's recipe:

First, ye boil yer water like ye're preparin' for a sea battle — hot and fierce! Then take yer coffee grounds (the darker the roast, the more it tastes like a stormy night at sea).

Pour that boilin' water over the grounds and let 'em steep like a ship anchored in port. Strain it through yer filter, and ARRR, ye've got yerself a proper cup of pirate fuel!

Now hoist the mainsail and set course for more questions! ⚓🏴‍☠️"""
    },
]

print(f"Created {len(PIRATE_DEMONSTRATIONS)} pirate demonstrations")

In [ ]:
# ================================================================
# TODO: CREATE YOUR OWN PERSONA!
# ================================================================
# 
# Uncomment and fill in YOUR demonstrations below.
# Try: Hungry Person, Shakespeare, Overly Enthusiastic Nerd, etc.
#
# MY_DEMONSTRATIONS = [
#     {
#         "prompt": "What is machine learning?",
#         "response": """YOUR STYLED RESPONSE HERE..."""
#     },
#     # Add 4-9 more examples...
# ]

# For this lab, we'll use the pirate demonstrations
DEMONSTRATIONS = MY_DEMONSTRATIONS

---

## 📝 TASK 2: Format Data for Training

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Prepare SFT dataset
def prepare_sft_data(demonstrations, tokenizer):
    sft_data = []
    for demo in demonstrations:
        messages = [
            {"role": "user", "content": demo["prompt"]},
            {"role": "assistant", "content": demo["response"]}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        sft_data.append({"text": text})
    return Dataset.from_list(sft_data)

# Prepare DITTO dataset
def prepare_ditto_data(demonstrations, tokenizer):
    ditto_data = []
    for demo in demonstrations:
        prompt_msg = [{"role": "user", "content": demo["prompt"]}]
        prompt_text = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)
        ditto_data.append({
            "prompt": prompt_text,
            "chosen": demo["response"]
        })
    return Dataset.from_list(ditto_data)

sft_dataset = prepare_sft_data(DEMONSTRATIONS, tokenizer)
ditto_dataset = prepare_ditto_data(DEMONSTRATIONS, tokenizer)

print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"DITTO dataset: {len(ditto_dataset)} examples")

---

## 📝 TASK 3: Train Standard SFT Model

In [ ]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM
)

# Add SFT adapter
model = get_peft_model(model, lora_config, adapter_name="sft")
model.set_adapter("sft")
print("SFT adapter added")

In [ ]:
# SFT output directory
sft_output_dir = "./outputs/exercise_sft"

# SFT Training
sft_config = SFTConfig(
    output_dir=sft_output_dir,
    max_seq_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=500,
    save_strategy="steps",
    bf16=True,
    packing=False,
    dataset_text_field="text",
)

sft_trainer_kwargs = {
    "model": model,
    "args": sft_config,
    "train_dataset": sft_dataset,
    TOKENIZER_KWARG: tokenizer,
    "callbacks": [EarlyStoppingCallback(threshold=1)],  # Stop when loss < 1
}
sft_trainer = SFTTrainer(**sft_trainer_kwargs)

print("\n" + "="*60)
print("Training Standard SFT...")
print("="*60)
sft_trainer.train()
print("SFT training complete!")

In [ ]:
# Save SFT adapter
save_adapter(model, tokenizer, sft_output_dir, "sft")
logger.info(f"SFT adapter saved to {sft_output_dir}")

---

## 📝 TASK 4: Train DITTO Model

In [ ]:
# Import the actual DITTOTrainer
# Add the ditto directory to path
sys.path.insert(0, './ditto')

from ditto_trainer import DITTOTrainer

print("Imported DITTOTrainer from ditto/ditto_trainer.py")

In [ ]:
logger.info("=" * 60)
logger.info("STAGE 2/2: DITTO Training (on top of SFT)")
logger.info("=" * 60)

# DITTO hyperparameters
ditto_output_dir = "./outputs/exercise_ditto"
DITTO_LEARNING_RATE = 5e-6
DITTO_MAX_STEPS = 30
DITTO_WARMUP_RATIO = 0.1
DITTO_BATCH_SIZE = 2
BETA = 0.1
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 512

# Add DITTO adapter and copy weights from SFT
model.add_adapter("ditto", lora_config)
model.set_adapter("ditto")
copy_adapter_weights("sft", "ditto", model)

print("DITTO adapter created and initialized from SFT weights")

In [ ]:
# Create training arguments for DITTO
@dataclass
class DITTOArgs(TrainingArguments):
    """Extended training arguments for DITTO."""
    frac_expert: float = 0.7
    frac_replay: float = 0.2
    frac_noisy: float = 0.1
    rescale_batch: int = 3
    resample_rate: int = 10
    bootstrap_count: int = 10
    reset_rate: int = -1

ditto_training_args = DITTOArgs(
    output_dir=ditto_output_dir,
    learning_rate=DITTO_LEARNING_RATE,
    max_steps=DITTO_MAX_STEPS,
    per_device_train_batch_size=DITTO_BATCH_SIZE,
    gradient_accumulation_steps=2,
    warmup_ratio=DITTO_WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=500,
    bf16=True,
    remove_unused_columns=False,
    # DITTO specific
    frac_expert=0.7,
    frac_replay=0.2,
    frac_noisy=0.1,
    rescale_batch=3,
    resample_rate=10,
    bootstrap_count=5,  # Reduced for demo
)

print(f"DITTO training args configured:")
print(f"  - Learning rate: {DITTO_LEARNING_RATE}")
print(f"  - Max steps: {DITTO_MAX_STEPS}")
print(f"  - Beta: {BETA}")
print(f"  - Comparison split: {ditto_training_args.frac_expert}/{ditto_training_args.frac_replay}/{ditto_training_args.frac_noisy}")

In [ ]:
# Initialize DITTOTrainer (exactly as in run_ditto.py)
ditto_trainer = DITTOTrainer(
    model=model,
    ref_adapter_name="sft",        # Reference model is the SFT adapter
    model_adapter_name="ditto",    # Training adapter
    args=ditto_training_args,
    beta=BETA,
    train_dataset=ditto_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    loss_type="sigmoid",
)

print("DITTOTrainer initialized")
print(f"  - Reference adapter: 'sft'")
print(f"  - Training adapter: 'ditto'")

In [ ]:
# Run DITTO training
logger.info("Starting DITTO training...")
train_result = ditto_trainer.train()

# Log metrics
metrics = train_result.metrics
metrics["train_samples"] = len(ditto_dataset)
ditto_trainer.log_metrics("train", metrics)
ditto_trainer.save_metrics("train", metrics)
ditto_trainer.save_state()

logger.info("*** DITTO Training complete ***")

In [ ]:
# Save DITTO adapter (from run_ditto.py)
model.delete_adapter("sft")  # Clean up
save_adapter(model, tokenizer, ditto_output_dir, "ditto")
logger.info(f"DITTO adapter saved to {ditto_output_dir}")

# Save config
if ditto_trainer.accelerator.is_main_process:
    ditto_trainer.model.config.use_cache = True
    ditto_trainer.model.config.save_pretrained(ditto_output_dir)

In [ ]:
# Summary
logger.info("=" * 60)
logger.info("Training Complete! Adapters saved:")
logger.info(f"  1. SFT:   {sft_output_dir}")
logger.info(f"  2. DITTO: {ditto_output_dir}")
logger.info("=" * 60)

---

## 📝 TASK 5: Compare Outputs!

In [ ]:
# Test prompts - DIFFERENT from training!
TEST_PROMPTS = [
    "What is photosynthesis?",
    "How do airplanes fly?",
    "Why do we dream?",
    "What is the stock market?",
]

In [ ]:
print("\n" + "="*80)
print("                         COMPARISON: SFT vs DITTO")
print("="*80)

for prompt in TEST_PROMPTS:
    print(f"\n{'─'*80}")
    print(f"PROMPT: {prompt}")
    print(f"{'─'*80}")
    
    # SFT response
    model.set_adapter("sft")
    sft_response = generate_response(model, tokenizer, prompt)
    print("\n🔹 SFT Response:")
    print(sft_response[:500])
    
    # DITTO response
    model.set_adapter("ditto")
    ditto_response = generate_response(model, tokenizer, prompt)
    print("\n🔸 DITTO Response:")
    print(ditto_response[:500])

---

## 📊 Analysis Questions

After running the comparison, answer these questions:

### 1. Style Consistency
- Which model maintains the target style more consistently?
- Does the signature phrase ("Now hoist the mainsail...") appear reliably?

### 2. Content Quality
- Are the explanations accurate and helpful?
- Does the style interfere with clarity?

### 3. Generalization
- How well do the models apply the style to NEW topics?
- Which model generalizes better beyond the training examples?

### 4. When to Use Which?
- In what scenarios would you prefer SFT?
- When would DITTO be worth the extra training time?

In [ ]:
# ================================================================
# YOUR ANALYSIS (Write your observations here)
# ================================================================

analysis = """
## My Observations:

### Style Consistency:
- SFT: [Your observation]
- DITTO: [Your observation]

### Content Quality:
- SFT: [Your observation]
- DITTO: [Your observation]

### Generalization:
- SFT: [Your observation]
- DITTO: [Your observation]

### Conclusion:
[Your overall conclusion about SFT vs DITTO for style alignment]
"""

print(analysis)

---

## Lab 2 Series Summary

| Lab | Topic | Key Concept |
|-----|-------|-------------|
| 2.1 | Data Preprocessing | Chat templates, format for SFT/DPO/KTO |
| 2.2 | SFT & DPO | Imitation vs. preference learning |
| 2.3 | DITTO | Few-shot personalized alignment |
| **2.4** | **Exercise** | **Build your own style dataset!** |

**The alignment journey**: Raw data → Formatted data → SFT (basics) → DPO/DITTO (refinement) → Personalized model

---

## References

- Shaikh et al. (2025). Aligning Language Models with Demonstrated Feedback. *ICLR 2025*.
- Rafailov et al. (2023). Direct Preference Optimization. *NeurIPS*.
- Souza, T. (2024). Taming LLMs. *GitHub*.